ARK-020 V4 — multi-skill continual cognition, operator build. Cell 0: mount → clone → pinned checkout → fail-closed scan.

In [ ]:
import json, subprocess, sys, os
PINNED_RUNNER_COMMIT = 'f4244e2fea135cd768a5b5de5890e7117406c9c2'
REPO = '/content/An-Ra-the-new-AGI-ark020v4'
print('STEP 1: mount Drive')
from google.colab import drive
try:
    drive.mount('/content/drive'); DRIVE_OK = True
except Exception as exc:
    DRIVE_OK = False; print('DRIVE MOUNT FAILED:', exc)
print('STEP 2: clone + detach-checkout the frozen executable')
if not os.path.exists(REPO):
    subprocess.run(['git','clone','--depth','50','--branch','Arkenstone','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
print('PINNED COMMIT OK:', head)
if not DRIVE_OK:
    raise SystemExit('SAFE ACTION: STOP — DRIVE UNAVAILABLE')
print('STEP 3: READ-ONLY resume scan (fail-closed)')
r = subprocess.run([sys.executable, os.path.join(REPO,'experiments/ARK-020-V4/run_ark020_v4.py'),
                    '--mode','scan','--drive-ok','True'], cwd=REPO)
assert r.returncode == 0, 'scan command failed — DO NOT RUN the campaign'
print('Scan completed. SAFE ACTION printed above (machine marker @@SCAN_JSON@@).')

In [ ]:
import subprocess, sys, torch, os
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before running.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
paths = ['experiments/ARK-020-V4/ark020_v4_core.py','experiments/ARK-020-V4/run_ark020_v4.py',
         'tests/test_ark020_v4.py','experiments/ARK-019/ark019_v4_core.py','experiments/ARK-019/run_ark019_v4.py',
         'experiments/ARK-019/run_ark019_v3.py','experiments/ARK-018/ark018_v3_common.py','experiments/ARK-018/ark018_v3_binding_fast.py']
for p in paths:
    subprocess.run([sys.executable,'-m','py_compile',os.path.join(REPO,p)], check=True)
print('compile gate: PASS on', len(paths), 'files')
r = subprocess.run([sys.executable,'-m','unittest','tests.test_ark020_v4'], cwd=REPO)
assert r.returncode == 0, 'test suite failed — DO NOT RUN the campaign'
print('ALL 39 TESTS PASS (incl. CLI contract, lifecycle retention, cadence, timebox, production smoke)')

In [ ]:
# Full campaign. Exact-resumable: rerun on a fresh T4 to continue.
import os, subprocess, sys
runner = os.path.join(REPO,'experiments/ARK-020-V4/run_ark020_v4.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
proc = subprocess.run([sys.executable, runner, '--mode','all'], cwd=REPO, env=env)
print('CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    print('Partial bundle written. Rerun cell 0 then this cell to resume exactly.')

In [ ]:
from pathlib import Path
root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
res = root / 'ARK-020_V4_RESULT.json'
if res.exists():
    print('ARK-020 V4 RESULT:'); print(res.read_text())
smoke = root / 'EXACT_RESUME_SMOKE_V4.json'
if smoke.exists():
    print('EXACT RESUME:', smoke.read_text()[:600])
z = root / 'ARKENSTONE_ARK020_V4_CONTINUAL_RESULTS.zip'
if not z.exists():
    z = root / 'ARKENSTONE_ARK020_V4_CONTINUAL_PARTIAL.zip'
if z.exists():
    print('ZIP:', z, z.stat().st_size, 'bytes')
    sha = Path(str(z) + '.sha256')
    if sha.exists(): print(sha.read_text())
    try:
        from google.colab import files; files.download(str(z))
    except Exception as exc:
        print('Manual download: Drive > genisis-arkenstone > ARK020_V4_CONTINUAL', exc)
else:
    print('No bundle yet — rerun cell 2 to resume.')